In [27]:
import pandas as pd
import numpy as np
from scipy.stats import poisson, binom

nd = pd.read_csv('/content/cleaned_cricket_data.csv')

nd['Strike_Rate_calc'] = (nd['Runs_Scored'] / nd['Balls_Faced'] * 100).replace([np.inf, -np.inf], np.nan).fillna(0)

metrics = ['Runs_Scored', 'Balls_Faced', 'Strike_Rate_calc']
summary = nd[metrics].agg(['mean', 'median', 'std'])
print("Descriptive Statistics:")
display(summary)

Descriptive Statistics:


,Runs_Scored,Balls_Faced,Strike_Rate_calc
mean,83.385754,74.603842,307.989693
median,82.000000,74.000000,110.597148
std,55.718040,43.142808,1001.935640


In [21]:
# 2. Vectorized calculation of Boundary and Dot ball percentage
if 'Balls_Faced' in nd.columns:
    if 'Total_Boundaries' in nd.columns:
        nd['Total_Boundaries_Percentage'] = (nd['Total_Boundaries'] / nd['Balls_Faced']).fillna(0) * 100
    if 'Dot_Balls' in nd.columns:
        nd['Dot_Ball_Percentage'] = (nd['Dot_Balls'] / nd['Balls_Faced']).fillna(0) * 100

cols_to_show = ['Balls_Faced']
for col in ['Total_Boundaries_Percentage', 'Dot_Ball_Percentage']:
    if col in nd.columns:
        cols_to_show.append(col)

# Display the results
print("Updated DataFrame with Percentages:")
display(nd[cols_to_show].head())

Updated DataFrame with Percentages:


,Balls_Faced,Total_Boundaries_Percentage
0,16,525.000000
1,33,393.939394
2,109,86.238532
3,74,108.108108
4,136,61.764706


In [20]:
if 'Dot_Balls' in nd.columns:
    print("Found 'Dot_Balls' column. Calculating total...")
    print(f"Total Dot Balls: {nd['Dot_Balls'].sum()}")
elif 'Runs_Scored' in nd.columns and 'Balls_Faced' in nd.columns:
    print("Searching for matches where 0 runs were scored (potential dot ball indicators)...")
    zero_run_matches = nd[nd['Runs_Scored'] == 0]
    display(zero_run_matches[['Batsman_Name', 'Runs_Scored', 'Balls_Faced']].head())
    print(f"Total potential dot balls: {len(zero_run_matches)}")
else:
    print("Required columns ('Runs_Scored'/'Balls_Faced') not found.")

Searching for matches where 0 runs were scored (potential dot ball indicators)...


,Batsman_Name,Runs_Scored,Balls_Faced
128,Root,0,47
165,Root,0,132
438,Root,0,143
1171,Kohli,0,54
1406,Rohit,0,65


Total potential dot balls: 19


In [15]:
import numpy as np

# Perform vectorized calculations for the whole DataFrame
if 'Balls_Faced' in nd.columns:
    # Calculate Boundary Percentage (assuming Total_Boundaries exists)
    if 'Total_Boundaries' in nd.columns:
        nd['Total_Boundaries_Percentage'] = (nd['Total_Boundaries'] / nd['Balls_Faced']).replace([np.inf, -np.inf], np.nan).fillna(0) * 100

    # Calculate Dot Ball Percentage (assuming Dot_Balls exists)
    if 'Dot_Balls' in nd.columns:
        nd['Dot_Ball_Percentage'] = (nd['Dot_Balls'] / nd['Balls_Faced']).replace([np.inf, -np.inf], np.nan).fillna(0) * 100

# Identify columns to display
result_cols = ['Batsman_Name', 'Runs_Scored', 'Balls_Faced']
for c in ['Total_Boundaries_Percentage', 'Dot_Ball_Percentage']:
    if c in nd.columns:
        result_cols.append(c)

print("Calculations completed for the entire dataset.")
display(nd[result_cols].head(10))

Calculations completed for the entire dataset.


,Batsman_Name,Runs_Scored,Balls_Faced,Total_Boundaries_Percentage
0,Williamson,160,16,525.000000
1,Kohli,32,33,393.939394
2,Kohli,43,109,86.238532
3,Williamson,167,74,108.108108
4,Smith,129,136,61.764706
5,Faf,62,38,242.105263
6,Williamson,78,66,127.272727
7,Kohli,44,119,102.521008
8,Root,105,131,30.534351
9,Smith,28,108,88.888889


In [9]:
# 3. Identify extreme outliers using percentiles (1% & 99%)
lower_p = nd['Runs_Scored'].quantile(0.01)
upper_p = nd['Runs_Scored'].quantile(0.99)

outliers = nd[(nd['Runs_Scored'] < lower_p) | (nd['Runs_Scored'] > upper_p)]
print(f"1st Percentile: {lower_p}, 99th Percentile: {upper_p}")
print(f"Number of extreme outliers identified: {len(outliers)}")
display(outliers.head())

1st Percentile: -9.0, 99th Percentile: 178.0
Number of extreme outliers identified: 65


,Match_ID,Innings,Batting_Team,Bowling_Team,Venue,Match_Format,Batsman_Name,Bowler_Name,Runs_Scored,Balls_Faced,...,Economy_Rate,Dismissal_Type,Player_Role,Match_Date_Clean,Match_Month,Match_Year,Batting_StrikeRate,Bowling_Economy,Match_Result,Strike_Rate_calc
28,45286,1,Sri Lanka,Sri Lanka,Kolkata,ODI,Nabi,Rabada,179,116,...,NaN,Run Out,Batsman,NaN,NaN,NaN,154.310345,NaN,Incomplete,154.310345
87,18297,2,New Zealand,Sri Lanka,Mumbai,ODI,Williamson,Bumrah,-10,19,...,NaN,Stumped,All-Rounder,2020-01-01,2020-01,2020.0,-52.631579,NaN,Incomplete,-52.631579
144,94282,1,Bangladesh,West Indies,London,Test,Nabi,Starc,-10,48,...,NaN,Stumped,Batsman,2021-05-10,2021-05,2021.0,-20.833333,NaN,Incomplete,-20.833333
273,11937,1,Australia,India,Melbourne,ODI,Russell,Malinga,-10,101,...,NaN,Hit Wicket,Batsman,NaN,NaN,NaN,-9.900990,NaN,Incomplete,-9.900990
387,10126,2,Sri Lanka,Bangladesh,Chennai,ODI,Root,Starc,-10,63,...,13.7,Run Out,Batsman,2021-05-10,2021-05,2021.0,-15.873016,-1.69136,Incomplete,-15.873016


In [8]:
# 4. Simulate 1000 random 'Runs Scored' using Poisson distribution
mu_runs = nd['Runs_Scored'].mean()
simulated_runs = np.random.poisson(mu_runs, 1000)

print(f"Simulation based on mean runs ({mu_runs:.2f}):")
print(f"First 20 simulated values: {simulated_runs[:20]}")

Simulation based on mean runs (83.39):
First 20 simulated values: [79 85 65 87 83 81 78 88 78 96 77 84 85 91 83 72 81 86 74 63]


In [10]:
# 5. Probability a batsman scores a six (binomial distribution)
if 'Sixes' in nd.columns and 'Balls_Faced' in nd.columns:
    total_sixes = nd['Sixes'].sum()
    total_balls = nd['Balls_Faced'].sum()
    p_six = total_sixes / total_balls

    n = 30 # Assume 30 balls faced
    k = 3  # Target 3 sixes
    prob_k = binom.pmf(k, n, p_six)

    print(f"Probability of hitting a six per ball: {p_six:.4f}")
    print(f"Probability of hitting exactly {k} sixes in {n} balls: {prob_k:.4f}")

Probability of hitting a six per ball: 0.0927
Probability of hitting exactly 3 sixes in 30 balls: 0.2339
